# Colab Baseline cot_structured 16384 vLLM Eval

Run the base `Qwen/Qwen3-4B-Thinking-2507` model with `cot_structured_only`, `vllm`, `bf16`, and `16384` generation tokens. This notebook intentionally does not load a LoRA/SFT adapter.

In [ ]:
!nvidia-smi

## Google Drive

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ModuleNotFoundError:
    IN_COLAB = False
    print('Not running in Colab; skipping Google Drive mount.')

## Repo Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/DarinAnthony/151B_SP26_Competition.git'
cwd = Path.cwd().resolve()

if IN_COLAB:
    REPO_DIR = Path('/content/151B_SP26_Competition')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif (cwd / 'pyproject.toml').exists():
    REPO_DIR = cwd
elif (cwd.parent / 'pyproject.toml').exists():
    REPO_DIR = cwd.parent
else:
    raise RuntimeError(f'Could not find repo root from {cwd}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo:', REPO_DIR)
print('Python:', sys.executable)

## Dependencies

Run this once per fresh Colab runtime. Restart the runtime if Colab asks after installing `vllm`, then rerun the setup cells.

In [ ]:
%pip install -q -U pip uv

!uv pip uninstall --system -y vllm torch torchvision torchaudio triton
!uv pip install --system --torch-backend=cu130 \
  "vllm==0.21.0" \
  "bitsandbytes>=0.46.1" \
  "jedi>=0.16" \
  "hydra-core>=1.3" \
  datasets accelerate transformers tqdm sympy pyyaml wandb


## Baseline Run Config

No adapter is configured in this notebook. Results are written directly to Google Drive when running in Colab.
Default eval target is the 226-row held-out split at `data/test.jsonl`, with `MAX_TOKENS=16384`.


In [ ]:
from pathlib import Path
import os

RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else '/cephfs/qwen_math_comp/eval_results',
))
DRIVE_RESULTS_DIR = Path(os.environ.get(
    'DRIVE_RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else str(RESULTS_DIR),
))
EVAL_DATA_PATH = Path(os.environ.get('EVAL_DATA_PATH', 'data/test.jsonl'))
RUN_NAME = os.environ.get('RUN_NAME', 'baseline_cot_structured_heldout_16384_vllm_4090')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '16384'))
RUNNER_MAX_MODEL_LEN = int(os.environ.get('RUNNER_MAX_MODEL_LEN', str(max(16384, MAX_TOKENS + 4096))))
PRIVATE_BATCH_SIZE = int(os.environ.get('PRIVATE_BATCH_SIZE', '25'))
PRIVATE_DATA_PATH = Path(os.environ.get(
    'PRIVATE_DATA_PATH',
    '/content/drive/MyDrive/qwen_math_comp/private.jsonl' if IN_COLAB else 'data/private.jsonl',
))
SUBMISSION_DIR = Path(os.environ.get(
    'SUBMISSION_DIR',
    f'/content/drive/MyDrive/qwen_math_comp/submissions/{RUN_NAME}' if IN_COLAB else str(RESULTS_DIR / 'submissions' / RUN_NAME),
))
SUBMISSION_PATH = Path(os.environ.get('SUBMISSION_PATH', str(SUBMISSION_DIR / 'private.jsonl')))
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '1')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')

os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES

if not EVAL_DATA_PATH.exists():
    raise FileNotFoundError(f'Eval data not found: {EVAL_DATA_PATH}')

print('EVAL_DATA_PATH =', EVAL_DATA_PATH)
print('RESULTS_DIR =', RESULTS_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('WANDB_NAME =', os.environ['WANDB_NAME'])
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MAX_MODEL_LEN =', os.environ['RUNNER_MAX_MODEL_LEN'])
print('PRIVATE_BATCH_SIZE =', PRIVATE_BATCH_SIZE)
print('PRIVATE_DATA_PATH =', PRIVATE_DATA_PATH)
print('SUBMISSION_PATH =', SUBMISSION_PATH)
print('RUNNER_MICRO_BATCH_SIZE =', os.environ['RUNNER_MICRO_BATCH_SIZE'])
print('RUNNER_PARALLEL_SAMPLES =', os.environ['RUNNER_PARALLEL_SAMPLES'])
print('WANDB_PROJECT =', os.environ.get('WANDB_PROJECT', '<unset; W&B disabled>'))

## Eval Helper

Equivalent held-out eval shell command over `data/test.jsonl`:

```bash
python -m experiments.prompt_engineering.src.eval \
  run=cot_structured_only \
  eval=full \
  eval.data_path=data/test.jsonl \
  eval.max_tokens=16384 \
  runner.engine=vllm \
  runner.quant=bf16 \
  results_dir=/content/drive/MyDrive/qwen_math_comp/eval_results \
  run_name=baseline_cot_structured_heldout_16384_vllm_4090
```

In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
import time

def build_eval_cmd(run_name, *, max_tokens, results_dir, data_path=EVAL_DATA_PATH, slice_indices=None):
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=cot_structured_only',
        'eval=full',
        f'eval.data_path={data_path}',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        'runner.adapter_path=null',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(4, f'eval.slice_indices=[{joined}]')
    return cmd

def backup_run_to_drive(run_name, *, results_dir=RESULTS_DIR, drive_results_dir=DRIVE_RESULTS_DIR):
    src = Path(results_dir) / run_name
    dst = Path(drive_results_dir) / run_name
    if not src.exists():
        print(f'No run directory to back up yet: {src}')
        return
    if src.resolve() == dst.resolve():
        print(f'Results are already on Drive: {src}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Backed up results to Google Drive: {dst}')

def run_eval(run_name, *, max_tokens=MAX_TOKENS, results_dir=RESULTS_DIR, data_path=EVAL_DATA_PATH, slice_indices=None):
    env = os.environ.copy()
    env.pop('ADAPTER_PATH', None)
    env['WANDB_NAME'] = WANDB_NAME
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['RUNNER_MAX_MODEL_LEN'] = str(RUNNER_MAX_MODEL_LEN)
    env['EVAL_METRICS_DIR'] = str(DRIVE_RESULTS_DIR)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
    env['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        results_dir=results_dir,
        data_path=data_path,
        slice_indices=slice_indices,
    )

    print('EVAL_DATA_PATH =', data_path)
    print('ADAPTER_PATH in subprocess =', env.get('ADAPTER_PATH'))
    print('WANDB_NAME =', env['WANDB_NAME'])
    print('DRIVE_RESULTS_DIR =', env['EVAL_METRICS_DIR'])
    print('RUNNER_MAX_MODEL_LEN =', env['RUNNER_MAX_MODEL_LEN'])
    print('RUNNER_MICRO_BATCH_SIZE =', env['RUNNER_MICRO_BATCH_SIZE'])
    print('RUNNER_PARALLEL_SAMPLES =', env['RUNNER_PARALLEL_SAMPLES'])
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))

    start = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

## 5-Item Smoke Test

In [ ]:
run_eval(
    f'{RUN_NAME}_smoke5',
    max_tokens=MAX_TOKENS,
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Standard Eval

In [ ]:
run_eval(
    RUN_NAME,
    max_tokens=MAX_TOKENS,
)

## Result Files

In [ ]:
results_root = Path(RESULTS_DIR)
if not results_root.exists():
    print(f'No results directory yet: {results_root}')
else:
    for path in sorted(results_root.glob('*'))[-10:]:
        print(path)
        for metrics_name in ['metrics.json', 'leaderboard.csv', 'leaderboard.jsonl']:
            metrics_path = path / metrics_name
            if metrics_path.exists():
                print('  ', metrics_path)

## Private Submission Generation

This path does not score answers because the private set has no gold labels. It writes the competition submission shape: one JSON object per line with `id`, `is_mcq`, and `response`.

In [ ]:
from shared.io import load_jsonl, save_jsonl
from shared.prompt_format import build_chat_messages
from shared.runner import load_model
from shared.schemas import RunnerCfg, SamplingCfg
from experiments.prompt_engineering.src.prompts import PROMPTS

def generate_private_submission(
    *,
    private_path=PRIVATE_DATA_PATH,
    output_path=SUBMISSION_PATH,
    max_tokens=MAX_TOKENS,
    batch_size=PRIVATE_BATCH_SIZE,
    slice_indices=None,
    resume=True,
):
    private_path = Path(private_path)
    output_path = Path(output_path)
    if not private_path.exists():
        raise FileNotFoundError(
            f'Private data not found: {private_path}. Set PRIVATE_DATA_PATH to the uploaded private JSONL.'
        )
    if private_path.resolve() == output_path.resolve():
        raise ValueError('Refusing to overwrite the private input file. Choose a different SUBMISSION_PATH.')

    items = load_jsonl(private_path)
    if slice_indices is not None:
        items = [items[i] for i in slice_indices if 0 <= i < len(items)]
    if not items:
        raise ValueError(f'No private items loaded from {private_path}.')
    batch_size = max(1, int(batch_size))

    completed = {}
    if resume and output_path.exists():
        for row in load_jsonl(output_path):
            if 'id' in row and 'response' in row:
                completed[int(row['id'])] = row
        print(f'Resuming from {output_path}: {len(completed)} completed rows found.')

    prompt = PROMPTS['cot_structured']
    sampling = SamplingCfg(name='greedy', temperature=0.0, top_p=1.0, top_k=-1, n_samples=1)
    runner_cfg = RunnerCfg(engine='vllm', quant='bf16', adapter_path=None)

    print(f'Loaded {len(items)} private items from {private_path}')
    global PRIVATE_HANDLE
    if 'PRIVATE_HANDLE' not in globals():
        print('Loading base model with no adapter...')
        PRIVATE_HANDLE = load_model(runner_cfg)
        print('Model loaded.')
    else:
        print('Reusing already-loaded base model.')
    pending = [item for item in items if int(item['id']) not in completed]
    print(f'Generating private responses in chunks of {batch_size}: {len(pending)} pending / {len(items)} total')

    total_chunks = (len(pending) + batch_size - 1) // batch_size
    for chunk_idx, start in enumerate(range(0, len(pending), batch_size), start=1):
        chunk = pending[start:start + batch_size]
        print(f'Chunk {chunk_idx}/{total_chunks}: generating {len(chunk)} items...')
        messages = [build_chat_messages(item, prompt) for item in chunk]
        outputs = PRIVATE_HANDLE.generate_batch(messages, sampling, max_tokens)
        for item, out in zip(chunk, outputs):
            completed[int(item['id'])] = {
                'id': int(item['id']),
                'is_mcq': bool(item.get('options')),
                'response': out.responses[0],
            }
        rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
        save_jsonl(rows, output_path)
        print(f'  saved {len(rows)}/{len(items)} rows to {output_path}')

    rows = [completed[int(item['id'])] for item in items if int(item['id']) in completed]
    save_jsonl(rows, output_path)
    print(f'Saved {len(rows)} private submission rows to {output_path}')
    return output_path

## 5-Item Private Generation Smoke

In [ ]:
generate_private_submission(
    output_path=SUBMISSION_PATH.with_name('private_smoke5.jsonl'),
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Private Submission

In [ ]:
# Run this after the private smoke file looks good.
generate_private_submission()

## Back Up Results to Google Drive

In [ ]:
# Run this after smoke or full eval. In Colab, RESULTS_DIR is already on Drive.
backup_run_to_drive(f'{RUN_NAME}_smoke5')
backup_run_to_drive(RUN_NAME)


## Optional Full Public Sanity Check

This includes the 900 train-split rows, so use it only to compare with older full-public logs.


In [ ]:
RUN_FULL_PUBLIC_SANITY = False

if RUN_FULL_PUBLIC_SANITY:
    run_eval('baseline_cot_structured_full_public_16384_vllm_4090', data_path=Path('data/public.jsonl'))
else:
    print('Set RUN_FULL_PUBLIC_SANITY = True to evaluate all 1,126 public rows.')
